# Data Augmentation and Regularization for CNNs

---

## Learning Objectives

By the end of this notebook, you will:

- Understand **why** data augmentation works (more diverse training data without collecting new samples)
- Use `torchvision.transforms` for common augmentations: flips, rotations, crops, color jitter, random erasing
- Visualize augmented samples to verify transformations look reasonable
- Apply augmentation **only to training data** (never to validation/test)
- Apply regularization techniques for CNNs: dropout, weight decay, batch norm
- Train with and without augmentation and compare results

## Prerequisites

- **DL100**: Neural network fundamentals (regularization concepts)
- **DL150**: PyTorch foundations
- **Notebook 01**: CNN basics (convolutions, pooling)
- **Notebook 02**: CNN on CIFAR-10/MNIST

## Table of Contents

1. [Setup and Imports](#1.-Setup-and-Imports)
2. [Why Data Augmentation?](#2.-Why-Data-Augmentation?)
3. [Common Augmentation Transforms](#3.-Common-Augmentation-Transforms)
4. [Visualizing Augmented Samples](#4.-Visualizing-Augmented-Samples)
5. [Critical: Augmentation on Train ONLY](#5.-Critical:-Augmentation-on-Train-ONLY)
6. [Regularization for CNNs](#6.-Regularization-for-CNNs)
7. [Training: With vs Without Augmentation](#7.-Training:-With-vs-Without-Augmentation)
8. [Common CV Pitfalls](#8.-Common-CV-Pitfalls)
9. [Common Mistakes and Debugging Tips](#9.-Common-Mistakes-and-Debugging-Tips)
10. [Exercises](#10.-Exercises)

---

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import os

set_global_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
print("Setup complete.")

---

## 2. Why Data Augmentation?

Data augmentation artificially expands the training set by applying **label-preserving transformations** to existing images.

### Key Benefits

- **More training data** without collecting or labeling new samples
- **Reduces overfitting** by presenting different views of the same object
- **Teaches invariances**: a cat is still a cat whether flipped, rotated, or slightly cropped
- Especially important when you have **limited training data**

### The Principle

If $\mathbf{x}$ is an image with label $y$, and $T$ is a transformation that preserves the label:

$$T(\mathbf{x}) \mapsto y \quad \text{(same label)}$$

The effective training set size grows from $N$ to $N \times |\text{augmentations}|$ (approximately).

### What NOT to do

- Do not apply augmentations that **change the label** (e.g., flipping digits 6/9)
- Do not augment validation or test data
- Do not apply overly aggressive augmentations that make images unrecognizable

---

## 3. Common Augmentation Transforms

| Transform | What it does | Typical use |
|-----------|-------------|-------------|
| `RandomHorizontalFlip` | Mirror left-right with prob $p$ | Natural images (not digits!) |
| `RandomRotation` | Rotate by random angle | General images |
| `RandomCrop` | Crop a random region (with optional padding) | Force model to use partial info |
| `ColorJitter` | Random brightness, contrast, saturation, hue | Color robustness |
| `RandomErasing` | Randomly erase a rectangular region | Occlusion robustness |
| `RandomAffine` | Random affine transformations (translate, scale, shear) | Geometric robustness |

In [ ]:
# Load CIFAR-10 with basic transforms first (for visualization)
data_dir = os.path.join("..", "..", "data")

# Basic transform: just convert to tensor
basic_transform = transforms.Compose([
    transforms.ToTensor(),
])

try:
    train_data_raw = datasets.CIFAR10(data_dir, train=True, download=True, transform=basic_transform)
    test_data_raw = datasets.CIFAR10(data_dir, train=False, download=True, transform=basic_transform)
    DATASET_NAME = "CIFAR-10"
    N_CHANNELS = 3
    IMG_SIZE = 32
    class_names = train_data_raw.classes
    print(f"Loaded {DATASET_NAME}: {len(train_data_raw)} train, {len(test_data_raw)} test")
except Exception as e:
    print(f"CIFAR-10 download failed: {e}")
    print("Falling back to MNIST.")
    train_data_raw = datasets.MNIST(data_dir, train=True, download=True, transform=basic_transform)
    test_data_raw = datasets.MNIST(data_dir, train=False, download=True, transform=basic_transform)
    DATASET_NAME = "MNIST"
    N_CHANNELS = 1
    IMG_SIZE = 28
    class_names = [str(i) for i in range(10)]
    print(f"Loaded {DATASET_NAME}: {len(train_data_raw)} train, {len(test_data_raw)} test")

print(f"Image size: {IMG_SIZE}x{IMG_SIZE}x{N_CHANNELS}")
print(f"Classes: {class_names}")

In [ ]:
# Demonstrate individual transforms
individual_transforms = {
    "Original": transforms.Compose([transforms.ToTensor()]),
    "HorizontalFlip": transforms.Compose([
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
    ]),
    "Rotation(30)": transforms.Compose([
        transforms.RandomRotation(30),
        transforms.ToTensor(),
    ]),
    "RandomCrop(28,pad=4)": transforms.Compose([
        transforms.RandomCrop(IMG_SIZE - 4, padding=4),
        transforms.Resize(IMG_SIZE),
        transforms.ToTensor(),
    ]),
    "ColorJitter": transforms.Compose([
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
        transforms.ToTensor(),
    ]),
    "RandomErasing": transforms.Compose([
        transforms.ToTensor(),
        transforms.RandomErasing(p=1.0, scale=(0.1, 0.3)),
    ]),
}

# Pick a sample image
from PIL import Image

sample_img_tensor, sample_label = train_data_raw[0]
# Convert tensor back to PIL for transforms
sample_pil = transforms.ToPILImage()(sample_img_tensor)

print(f"Sample image class: {class_names[sample_label]}")

---

## 4. Visualizing Augmented Samples

Always visualize your augmentations to verify they:
- Produce reasonable-looking images
- Preserve the label semantics
- Are not too aggressive

In [ ]:
# Visualize each transform applied to the same image
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for ax, (name, tfm) in zip(axes.flatten(), individual_transforms.items()):
    # Apply transform
    if name == "RandomErasing":
        img_tensor = tfm(sample_pil)
    else:
        img_tensor = tfm(sample_pil)

    # Convert to displayable format (C, H, W) -> (H, W, C)
    if N_CHANNELS == 1:
        ax.imshow(img_tensor.squeeze(), cmap="gray")
    else:
        ax.imshow(img_tensor.permute(1, 2, 0).clamp(0, 1))

    ax.set_title(name, fontsize=11)
    ax.axis("off")

plt.suptitle(f"Individual Augmentations on a '{class_names[sample_label]}' image", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Show multiple random augmentations of the same image
combined_augmentation = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomCrop(IMG_SIZE, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.3, scale=(0.05, 0.15)),
])

fig, axes = plt.subplots(2, 5, figsize=(14, 6))

for ax in axes.flatten():
    img_tensor = combined_augmentation(sample_pil)
    if N_CHANNELS == 1:
        ax.imshow(img_tensor.squeeze(), cmap="gray")
    else:
        ax.imshow(img_tensor.permute(1, 2, 0).clamp(0, 1))
    ax.axis("off")

plt.suptitle("10 Random Augmentations of the Same Image", fontsize=13)
plt.tight_layout()
plt.show()

print("Each training epoch shows a different augmented version of each image.")
print("This is equivalent to having a much larger dataset.")

---

## 5. Critical: Augmentation on Train ONLY

**This is the most important rule of data augmentation:**

$$\text{Augmentation} \in \{\text{train transforms}\} \quad \text{but} \quad \text{Augmentation} \notin \{\text{val/test transforms}\}$$

### Why?

- Validation/test should mimic real-world deployment conditions
- Augmented val/test images give **noisy, unreliable** metrics
- Deterministic evaluation is essential for comparing models

### Correct Pattern

```python
# TRAIN transforms: augmentation + normalization
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

# VAL/TEST transforms: normalization ONLY (no augmentation!)
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
```

In [ ]:
# Define SEPARATE transforms for train vs eval
if DATASET_NAME == "CIFAR-10":
    MEAN = (0.4914, 0.4822, 0.4465)
    STD = (0.2470, 0.2435, 0.2616)
else:  # MNIST
    MEAN = (0.1307,)
    STD = (0.3081,)

# TRAIN: augmentation + normalization
train_transform_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomCrop(IMG_SIZE, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.2, scale=(0.05, 0.15)),
])

# TRAIN without augmentation (for comparison)
train_transform_no_aug = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# EVAL: normalization ONLY (no augmentation!)
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

print("Transforms defined:")
print(f"  Train (with aug):    {len(train_transform_aug.transforms)} transforms")
print(f"  Train (no aug):      {len(train_transform_no_aug.transforms)} transforms")
print(f"  Eval (val/test):     {len(eval_transform.transforms)} transforms")

---

## 6. Regularization for CNNs

Data augmentation is one form of regularization. Other common techniques:

### Dropout

Randomly zeros activations during training with probability $p$:

$$h'_i = \begin{cases} 0 & \text{with probability } p \\ h_i / (1-p) & \text{with probability } 1-p \end{cases}$$

- Prevents co-adaptation of neurons
- Typically used after fully-connected layers (less common in conv layers)
- Common values: $p = 0.2$ to $0.5$

### Weight Decay (L2 Regularization)

Adds a penalty proportional to squared weight magnitudes:

$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{data}} + \lambda \sum_{i} w_i^2$$

- Encourages smaller weights
- Implemented via `weight_decay` parameter in optimizer
- Typical values: $10^{-4}$ to $10^{-2}$

### Batch Normalization

Not purely regularization, but has a regularizing effect:

- Normalizes layer activations per mini-batch
- Adds noise (mini-batch statistics vary), which acts as mild regularization
- Enables higher learning rates and faster convergence

In [ ]:
class CNN(nn.Module):
    """CNN with configurable regularization."""

    def __init__(self, n_channels=3, n_classes=10, dropout=0.0, use_batchnorm=False):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(n_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32) if use_batchnorm else nn.Identity(),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32) if use_batchnorm else nn.Identity(),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64) if use_batchnorm else nn.Identity(),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64) if use_batchnorm else nn.Identity(),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        # Compute the flattened feature size
        feat_size = 64 * (IMG_SIZE // 4) * (IMG_SIZE // 4)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(feat_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# Show model summary
model_example = CNN(n_channels=N_CHANNELS, n_classes=10, dropout=0.3, use_batchnorm=True)
n_params = sum(p.numel() for p in model_example.parameters())
print(f"CNN parameters: {n_params:,}")
print(model_example)

---

## 7. Training: With vs Without Augmentation

We train the same CNN architecture in two settings:

1. **No augmentation**: basic transforms only
2. **With augmentation**: full augmentation pipeline

Both use the same eval transform for validation.

In [ ]:
def train_cnn(model, train_loader, val_loader, epochs=20, lr=1e-3, weight_decay=0.0):
    """Train CNN and return history."""
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        # Train
        model.train()
        losses, correct, total = [], 0, 0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            logits = model(X_b)
            loss = loss_fn(logits, y_b)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            correct += (logits.argmax(-1) == y_b).sum().item()
            total += y_b.size(0)

        history["train_loss"].append(np.mean(losses))
        history["train_acc"].append(correct / total)

        # Validate
        model.eval()
        losses, correct, total = [], 0, 0
        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                logits = model(X_b)
                losses.append(loss_fn(logits, y_b).item())
                correct += (logits.argmax(-1) == y_b).sum().item()
                total += y_b.size(0)

        history["val_loss"].append(np.mean(losses))
        history["val_acc"].append(correct / total)

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{epochs} | "
                  f"Train Acc: {history['train_acc'][-1]:.4f} | "
                  f"Val Acc: {history['val_acc'][-1]:.4f}")

    return history

In [ ]:
# Create datasets with different transforms
if DATASET_NAME == "CIFAR-10":
    train_ds_aug = datasets.CIFAR10(data_dir, train=True, download=True, transform=train_transform_aug)
    train_ds_no_aug = datasets.CIFAR10(data_dir, train=True, download=True, transform=train_transform_no_aug)
    val_ds = datasets.CIFAR10(data_dir, train=False, download=True, transform=eval_transform)
else:
    train_ds_aug = datasets.MNIST(data_dir, train=True, download=True, transform=train_transform_aug)
    train_ds_no_aug = datasets.MNIST(data_dir, train=True, download=True, transform=train_transform_no_aug)
    val_ds = datasets.MNIST(data_dir, train=False, download=True, transform=eval_transform)

# Use a subset for faster training (optional -- remove for full training)
SUBSET_SIZE = 5000
set_global_seed(42)
indices = torch.randperm(len(train_ds_aug))[:SUBSET_SIZE].tolist()

train_sub_aug = Subset(train_ds_aug, indices)
train_sub_no_aug = Subset(train_ds_no_aug, indices)

BATCH_SIZE = 64
train_loader_aug = DataLoader(train_sub_aug, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
train_loader_no_aug = DataLoader(train_sub_no_aug, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=0)

print(f"Training subset: {SUBSET_SIZE} samples")
print(f"Validation: {len(val_ds)} samples")
print(f"Batch size: {BATCH_SIZE}")

In [ ]:
# Train without augmentation
EPOCHS = 20

print("Training WITHOUT augmentation:")
print("=" * 50)
set_global_seed(42)
model_no_aug = CNN(n_channels=N_CHANNELS, n_classes=10, dropout=0.0, use_batchnorm=False).to(device)
history_no_aug = train_cnn(model_no_aug, train_loader_no_aug, val_loader, epochs=EPOCHS, lr=1e-3)

In [ ]:
# Train with augmentation + regularization
print("\nTraining WITH augmentation + dropout + weight decay + batch norm:")
print("=" * 50)
set_global_seed(42)
model_aug = CNN(n_channels=N_CHANNELS, n_classes=10, dropout=0.3, use_batchnorm=True).to(device)
history_aug = train_cnn(model_aug, train_loader_aug, val_loader, epochs=EPOCHS, lr=1e-3, weight_decay=1e-4)

In [ ]:
# Compare learning curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, EPOCHS + 1)

# Loss
ax1.plot(ep, history_no_aug["train_loss"], label="No Aug - Train", linestyle="-", color="C0")
ax1.plot(ep, history_no_aug["val_loss"], label="No Aug - Val", linestyle="--", color="C0")
ax1.plot(ep, history_aug["train_loss"], label="With Aug - Train", linestyle="-", color="C1")
ax1.plot(ep, history_aug["val_loss"], label="With Aug - Val", linestyle="--", color="C1")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("Loss: With vs Without Augmentation")
ax1.legend(); ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(ep, history_no_aug["train_acc"], label="No Aug - Train", linestyle="-", color="C0")
ax2.plot(ep, history_no_aug["val_acc"], label="No Aug - Val", linestyle="--", color="C0")
ax2.plot(ep, history_aug["train_acc"], label="With Aug - Train", linestyle="-", color="C1")
ax2.plot(ep, history_aug["val_acc"], label="With Aug - Val", linestyle="--", color="C1")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy: With vs Without Augmentation")
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

gap_no_aug = history_no_aug["train_acc"][-1] - history_no_aug["val_acc"][-1]
gap_aug = history_aug["train_acc"][-1] - history_aug["val_acc"][-1]

print(f"\nGeneralization gap (train - val acc):")
print(f"  No augmentation:   {gap_no_aug:.4f}")
print(f"  With augmentation: {gap_aug:.4f}")
print(f"\nFinal val accuracy:")
print(f"  No augmentation:   {history_no_aug['val_acc'][-1]:.4f}")
print(f"  With augmentation: {history_aug['val_acc'][-1]:.4f}")

---

## 8. Common CV Pitfalls

### Normalization Mismatch

If training uses `Normalize(mean, std)` but inference does not, predictions will be incorrect.
Always save and reuse the exact same normalization parameters.

### CHW vs HWC

- **PyTorch** expects images in `(C, H, W)` format (channels first)
- **NumPy/PIL/OpenCV** typically uses `(H, W, C)` format (channels last)
- `transforms.ToTensor()` handles this conversion automatically from PIL images
- Manual conversion: `tensor.permute(2, 0, 1)` for HWC->CHW

### Augmentation Leakage

- Applying augmentation to the validation/test set
- Applying augmentation before train/val split (augmented copies could appear in both sets)
- Using test-time augmentation without understanding it (TTA is a valid but distinct technique)

In [ ]:
# Demonstrate CHW vs HWC
sample_tensor, _ = train_data_raw[0]
print(f"PyTorch tensor shape (CHW): {sample_tensor.shape}")

# Convert to numpy HWC for display
if N_CHANNELS == 1:
    sample_numpy = sample_tensor.squeeze().numpy()
    print(f"NumPy array shape (HW):    {sample_numpy.shape}")
else:
    sample_numpy = sample_tensor.permute(1, 2, 0).numpy()
    print(f"NumPy array shape (HWC):   {sample_numpy.shape}")

# Common mistake: forgetting to convert
print("\nCommon error if you forget: RuntimeError about expected 4D input")
print("Solution: Ensure your tensor is (N, C, H, W) for PyTorch models.")

---

## 9. Common Mistakes and Debugging Tips

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Augmentation on val/test | Noisy validation metrics | Use separate transforms: augmented for train, plain for eval |
| Normalization mismatch train/inference | Model performs terribly on new data | Save and reuse exact `mean`/`std` values |
| CHW vs HWC confusion | Shape errors or garbled images | Use `transforms.ToTensor()` or `permute()` |
| Too aggressive augmentation | Model cannot learn (high train loss) | Visualize augmented images, reduce intensity |
| Horizontal flip on digits | 6 becomes 9 | Choose augmentations appropriate to your domain |
| Forgetting `model.eval()` | Dropout/BN active during evaluation | Always toggle `train()`/`eval()` |
| No augmentation on small datasets | Severe overfitting | Add augmentation, it helps most with small data |
| Weight decay too high | Underfitting (model too constrained) | Typical values: $10^{-4}$ to $10^{-3}$ |

---

## 10. Exercises

### Exercise 1: Design an Augmentation Pipeline

Create a custom augmentation pipeline for the dataset. Experiment with:

- Different rotation angles (5, 15, 30 degrees)
- Different crop padding (2, 4, 8)
- Different ColorJitter intensities
- Adding `RandomAffine` with translation

Visualize your augmented samples and train a CNN. Does your pipeline beat the default?

In [ ]:
# ===== EXERCISE 1: Your code here =====
# custom_augmentation = transforms.Compose([
#     transforms.RandomAffine(degrees=15, translate=(0.1, 0.1)),
#     transforms.RandomCrop(IMG_SIZE, padding=6),
#     transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
#     transforms.ToTensor(),
#     transforms.Normalize(MEAN, STD),
# ])
# ...
pass

### Exercise 2: Regularization Ablation Study

Train the CNN with different combinations of regularization:

1. No regularization
2. Dropout only (0.3)
3. Weight decay only (1e-4)
4. Batch norm only
5. All combined

Use the augmented training set for all. Compare val accuracy.

In [ ]:
# ===== EXERCISE 2: Your code here =====
# reg_configs = {
#     "None": {"dropout": 0.0, "use_batchnorm": False, "weight_decay": 0.0},
#     "Dropout": {"dropout": 0.3, "use_batchnorm": False, "weight_decay": 0.0},
#     "WeightDecay": {"dropout": 0.0, "use_batchnorm": False, "weight_decay": 1e-4},
#     "BatchNorm": {"dropout": 0.0, "use_batchnorm": True, "weight_decay": 0.0},
#     "All": {"dropout": 0.3, "use_batchnorm": True, "weight_decay": 1e-4},
# }
pass

### Exercise 1 -- Solution

In [ ]:
custom_augmentation = transforms.Compose([
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.RandomCrop(IMG_SIZE, padding=6),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.25, scale=(0.05, 0.2)),
])

# Visualize
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax in axes.flatten():
    img_tensor = custom_augmentation(sample_pil)
    # Denormalize for display
    mean_t = torch.tensor(MEAN).view(-1, 1, 1)
    std_t = torch.tensor(STD).view(-1, 1, 1)
    img_display = img_tensor * std_t + mean_t
    if N_CHANNELS == 1:
        ax.imshow(img_display.squeeze(), cmap="gray")
    else:
        ax.imshow(img_display.permute(1, 2, 0).clamp(0, 1))
    ax.axis("off")
plt.suptitle("Custom Augmentation Pipeline", fontsize=13)
plt.tight_layout()
plt.show()

# Train with custom augmentation
if DATASET_NAME == "CIFAR-10":
    train_ds_custom = datasets.CIFAR10(data_dir, train=True, download=True, transform=custom_augmentation)
else:
    train_ds_custom = datasets.MNIST(data_dir, train=True, download=True, transform=custom_augmentation)

train_sub_custom = Subset(train_ds_custom, indices)
train_loader_custom = DataLoader(train_sub_custom, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

print("\nTraining with custom augmentation:")
set_global_seed(42)
model_custom = CNN(n_channels=N_CHANNELS, n_classes=10, dropout=0.3, use_batchnorm=True).to(device)
history_custom = train_cnn(model_custom, train_loader_custom, val_loader, epochs=EPOCHS, lr=1e-3, weight_decay=1e-4)

print(f"\nCustom aug val acc:  {max(history_custom['val_acc']):.4f}")
print(f"Default aug val acc: {max(history_aug['val_acc']):.4f}")

### Exercise 2 -- Solution

In [ ]:
reg_configs = {
    "None": {"dropout": 0.0, "use_batchnorm": False, "weight_decay": 0.0},
    "Dropout": {"dropout": 0.3, "use_batchnorm": False, "weight_decay": 0.0},
    "WeightDecay": {"dropout": 0.0, "use_batchnorm": False, "weight_decay": 1e-4},
    "BatchNorm": {"dropout": 0.0, "use_batchnorm": True, "weight_decay": 0.0},
    "All": {"dropout": 0.3, "use_batchnorm": True, "weight_decay": 1e-4},
}

reg_histories = {}
for name, cfg in reg_configs.items():
    print(f"\nConfig: {name}")
    set_global_seed(42)
    model = CNN(
        n_channels=N_CHANNELS, n_classes=10,
        dropout=cfg["dropout"], use_batchnorm=cfg["use_batchnorm"]
    ).to(device)
    h = train_cnn(model, train_loader_aug, val_loader,
                  epochs=EPOCHS, lr=1e-3, weight_decay=cfg["weight_decay"])
    reg_histories[name] = h
    print(f"  Best Val Acc: {max(h['val_acc']):.4f}")

# Plot comparison
fig, ax = plt.subplots(figsize=(10, 6))
ep = range(1, EPOCHS + 1)
for name, h in reg_histories.items():
    ax.plot(ep, h["val_acc"], label=name, linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation Accuracy")
ax.set_title("Regularization Ablation Study")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nSummary:")
for name, h in reg_histories.items():
    print(f"  {name:15s}: Best Val Acc = {max(h['val_acc']):.4f}")

---

**Next notebook:** [04 -- Transfer Learning: ResNet Feature Extractor](04_Transfer_Learning_ResNet_FeatureExtractor.ipynb)